In [1]:
print("Hello, world!")

Hello, world!


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow_datasets as tfds
import tensorflow as tf
import os
import time
import matplotlib.pyplot as plt
from IPython.display import clear_output
AUTOTUNE = tf.data.AUTOTUNE
import numpy as np
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models

In [ ]:
def load_npy_files(directory):
    file_paths = [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.npy')]

    source_data = []
    target_data = []

    for file_path in file_paths:
        data = np.load(file_path)
        source_data.append(data[0])
        target_data.append(data[1])

    return source_data, target_data


def deMean(x, mean, std):
    return (x - mean)/std

def deMean_after(x, mean_value, std_value):
    x_array = np.array(x, dtype=np.float64)
    valid_mask = ~np.isnan(x_array)
    result = np.where(valid_mask, (x_array - mean_value)/std_value, x_array)
    result = np.nan_to_num(result, nan=0.0)
    return result


def preprocess_signal_source(signal, mean_value, std_value):
    signal = deMean_after(signal, mean_value, std_value)
    return tf.convert_to_tensor(signal, dtype=tf.float32)

def preprocess_signal_target(signal, mean_value, std_value):
    signal = deMean(signal, mean_value, std_value)
    return tf.convert_to_tensor(signal, dtype=tf.float32)


def split_dataset(source_data, target_data, train_split=0.8):
    dataset_size = len(source_data)
    train_size = int(dataset_size * train_split)

    indices = np.random.permutation(dataset_size)
    train_indices = indices[:train_size]
    test_indices = indices[train_size:]

    train_source = [source_data[i] for i in train_indices]
    train_target = [target_data[i] for i in train_indices]
    test_source = [source_data[i] for i in test_indices]
    test_target = [target_data[i] for i in test_indices]

    return train_source, train_target, test_source, test_target

def prepare_dataset(source_data, target_data, batch_size=16, shuffle_buffer=1000):

    processed_source_data = []
    processed_target_data = []

    for source, target in zip(source_data, target_data):
        mean_value = np.nanmean(target)
        std_value = np.nanstd(target)

        processed_source = preprocess_signal_source(source, mean_value, std_value)
        processed_target = preprocess_signal_target(target, mean_value, std_value)

        processed_source_data.append(processed_source)
        processed_target_data.append(processed_target)

    dataset = tf.data.Dataset.from_tensor_slices((processed_source_data, processed_target_data))
    dataset = dataset.shuffle(shuffle_buffer).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset


In [ ]:
downsample_signal_length = 512

def downsample_signal(signal, target_length=downsample_signal_length):
    factor = len(signal) // target_length
    return np.mean(signal.reshape(-1, factor), axis=1)

def upsample_signal(signal, original_length=4092):
    x_original = np.linspace(0, 1, len(signal))
    x_target = np.linspace(0, 1, original_length)
    return np.interp(x_target, x_original, signal)

def preprocess_signal_downsampled(signal, mean_value, std_value):
    signal = deMean_after(signal, mean_value, std_value)
    downsampled = downsample_signal(signal)
    return tf.convert_to_tensor(downsampled, dtype=tf.float32)

def preprocess_signal_target_downsampled(signal, mean_value, std_value):
    signal = deMean(signal, mean_value, std_value)
    downsampled = downsample_signal(signal)
    return tf.convert_to_tensor(downsampled, dtype=tf.float32)

def prepare_downsampled_dataset(source_data, target_data, batch_size=16, shuffle_buffer=1000):
    processed_source_data = []
    processed_target_data = []

    for source, target in zip(source_data, target_data):
        mean_value = np.nanmean(target)
        std_value = np.nanstd(target)

        processed_source = preprocess_signal_downsampled(source, mean_value, std_value)
        processed_target = preprocess_signal_target_downsampled(target, mean_value, std_value)

        processed_source_data.append(processed_source)
        processed_target_data.append(processed_target)

    dataset = tf.data.Dataset.from_tensor_slices((processed_source_data, processed_target_data))
    dataset = dataset.shuffle(shuffle_buffer).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
dataset_dir = "/content/dataset/inpainting"
batch_size = 1
source_data, target_data = load_npy_files(dataset_dir)

train_source, train_target, test_source, test_target = split_dataset(source_data, target_data, train_split=0.8)

In [ ]:
train_dataset_downsampled = prepare_downsampled_dataset(train_source, train_target, batch_size=batch_size)
test_dataset_downsampled = prepare_downsampled_dataset(test_source, test_target, batch_size=batch_size)

In [ ]:
def build_conv_autoencoder(input_dim):
    kernel_size = 31
    pool_size = 8
    input_signal = layers.Input(shape=(input_dim, 1))

    encoded = layers.Conv1D(64, kernel_size=kernel_size, activation='relu', padding='same')(input_signal)
    encoded = layers.MaxPooling1D(pool_size=4, padding='same')(encoded)
    encoded = layers.Conv1D(32, kernel_size=kernel_size, activation='relu', padding='same')(encoded)
    encoded = layers.MaxPooling1D(pool_size=4, padding='same')(encoded)

    decoded = layers.Conv1D(32, kernel_size=kernel_size, activation='relu', padding='same')(encoded)
    decoded = layers.UpSampling1D(size=4)(decoded)
    decoded = layers.Conv1D(64, kernel_size=kernel_size, activation='relu', padding='same')(decoded)
    decoded = layers.UpSampling1D(size=4)(decoded)
    decoded = layers.Conv1D(1, kernel_size=kernel_size, activation='linear', padding='same')(decoded)

    autoencoder = models.Model(inputs=input_signal, outputs=decoded)
    return autoencoder

In [ ]:
def build_conv_autoencoder(input_dim):
    input_signal = layers.Input(shape=(input_dim, 1))

    # Encoder
    encoded = layers.Conv1D(128, kernel_size=15, activation='relu', padding='same')(input_signal)
    encoded = layers.MaxPooling1D(pool_size=2, padding='same')(encoded)
    encoded = layers.Conv1D(64, kernel_size=15, activation='relu', padding='same')(encoded)
    encoded = layers.MaxPooling1D(pool_size=2, padding='same')(encoded)
    encoded = layers.Conv1D(32, kernel_size=15, activation='relu', padding='same')(encoded)
    encoded = layers.MaxPooling1D(pool_size=2, padding='same')(encoded)

    # Decoder
    decoded = layers.Conv1D(32, kernel_size=15, activation='relu', padding='same')(encoded)
    decoded = layers.UpSampling1D(size=2)(decoded)
    decoded = layers.Conv1D(64, kernel_size=15, activation='relu', padding='same')(decoded)
    decoded = layers.UpSampling1D(size=2)(decoded)
    decoded = layers.Conv1D(128, kernel_size=15, activation='relu', padding='same')(decoded)
    decoded = layers.UpSampling1D(size=2)(decoded)
    decoded = layers.Conv1D(1, kernel_size=15, activation='linear', padding='same')(decoded)

    autoencoder = models.Model(inputs=input_signal, outputs=decoded)
    return autoencoder


In [ ]:
def build_advanced_conv_autoencoder(input_dim):
    input_signal = layers.Input(shape=(input_dim, 1))

    # Encoder
    encoded = layers.Conv1D(128, kernel_size=31, activation='relu', padding='same')(input_signal)
    encoded = layers.MaxPooling1D(pool_size=2, padding='same')(encoded)
    encoded = layers.Conv1D(64, kernel_size=15, activation='relu', padding='same')(encoded)
    encoded = layers.MaxPooling1D(pool_size=2, padding='same')(encoded)
    encoded = layers.Conv1D(32, kernel_size=7, activation='relu', padding='same', dilation_rate=2)(encoded)
    encoded = layers.MaxPooling1D(pool_size=2, padding='same')(encoded)
    encoded = layers.Dropout(0.2)(encoded)

    # Decoder
    decoded = layers.Conv1D(32, kernel_size=7, activation='relu', padding='same')(encoded)
    decoded = layers.UpSampling1D(size=2)(decoded)
    decoded = layers.Conv1D(64, kernel_size=15, activation='relu', padding='same')(decoded)
    decoded = layers.UpSampling1D(size=2)(decoded)
    decoded = layers.Conv1D(128, kernel_size=31, activation='relu', padding='same')(decoded)
    decoded = layers.UpSampling1D(size=2)(decoded)
    decoded = layers.Conv1D(1, kernel_size=15, activation='linear', padding='same')(decoded)

    autoencoder = models.Model(inputs=input_signal, outputs=decoded)
    return autoencoder


In [ ]:
def masked_mse_loss(y_true, y_pred):
    mask = tf.cast(~tf.math.is_nan(y_true), tf.float32)
    y_true = tf.where(tf.math.is_nan(y_true), tf.zeros_like(y_true), y_true)
    return tf.reduce_mean(tf.square((y_true - y_pred) * mask))


In [ ]:
signal_length = 512
autoencoder = build_advanced_conv_autoencoder(signal_length)
autoencoder.compile(optimizer='adam', loss=masked_mse_loss)

In [ ]:
autoencoder.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)           │ (None, 512, 1)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_29 (Conv1D)                   │ (None, 512, 128)            │           4,096 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_12 (MaxPooling1D)      │ (None, 256, 128)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_30 (Conv1D)                   │ (None, 256, 64)             │         122,944 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_13 (MaxPooling1D)      │ (None, 128, 64)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_31 (Conv1D)                   │ (None, 128, 32)             │          14,368 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_14 (MaxPooling1D)      │ (None, 64, 32)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64, 32)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_32 (Conv1D)                   │ (None, 64, 32)              │           7,200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ up_sampling1d_12 (UpSampling1D)      │ (None, 128, 32)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_33 (Conv1D)                   │ (None, 128, 64)             │          30,784 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ up_sampling1d_13 (UpSampling1D)      │ (None, 256, 64)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_34 (Conv1D)                   │ (None, 256, 128)            │         254,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ up_sampling1d_14 (UpSampling1D)      │ (None, 512, 128)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_35 (Conv1D)                   │ (None, 512, 1)              │           1,921 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 435,393 (1.66 MB)

 Trainable params: 435,393 (1.66 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = autoencoder.fit(train_dataset_downsampled, validation_data=test_dataset_downsampled, epochs=5)

Epoch 1/5
33280/33280 ━━━━━━━━━━━━━━━━━━━━ 102s 3ms/step - loss: 0.0887 - val_loss: 0.0847
Epoch 2/5
33280/33280 ━━━━━━━━━━━━━━━━━━━━ 139s 3ms/step - loss: 0.0789 - val_loss: 0.0737
Epoch 3/5
33280/33280 ━━━━━━━━━━━━━━━━━━━━ 144s 3ms/step - loss: 0.0755 - val_loss: 0.0767
Epoch 4/5
33280/33280 ━━━━━━━━━━━━━━━━━━━━ 141s 3ms/step - loss: 0.0725 - val_loss: 0.0756
Epoch 5/5
33280/33280 ━━━━━━━━━━━━━━━━━━━━ 103s 3ms/step - loss: 0.0707 - val_loss: 0.0963


In [ ]:
evaluation = autoencoder.evaluate(test_dataset_downsampled)
print(f"Test Loss: {evaluation}")

8320/8320 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step - loss: 0.0902
Test Loss: 0.08866589516401291


In [ ]:
import plotly.graph_objects as go

def visualize_predictions_plotly(model, dataset, num_samples=5):

    for source, target in dataset.take(num_samples):
        predictions = model.predict(source)
        source = source.numpy()
        target = target.numpy()

        for i in range(min(num_samples, len(source))):
            fig = go.Figure()

            fig.add_trace(go.Scatter(
                y=source[i],
                mode='lines',
                name='Source (With NaN)',
                line=dict(color='blue')
            ))

            fig.add_trace(go.Scatter(
                y=target[i],
                mode='lines',
                name='Ground Truth',
                line=dict(color='green')
            ))

            fig.add_trace(go.Scatter(
                y=tf.reshape(predictions[i], [-1]),
                mode='lines',
                name='Inpainted Signal',
                line=dict(color='red')
            ))

            fig.update_layout(
                title=f"Signal Inpainting - Sample {i+1}",
                xaxis_title="Time",
                yaxis_title="Value",
                legend=dict(x=0, y=1, bgcolor="rgba(255,255,255,0)", bordercolor="rgba(0,0,0,0)")
            )

            fig.show()


In [ ]:
visualize_predictions_plotly(autoencoder, test_dataset_downsampled, num_samples=20)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 684ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


In [ ]:
model_path = "inpainter_v512_2.keras"
autoencoder.save(model_path)
print(f"Model saved at: {model_path}")


Model saved at: inpainter_v512_2.keras
